In [1]:
import pandas as pd
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

CONTENT_MOVIES_PATH = PROCESSED_DATA_DIR / "content_movies.csv"

In [3]:
movies = pd.read_csv(CONTENT_MOVIES_PATH)
movies.head()

,id,title,tags,vote_average,vote_count,release_date,runtime
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...",7.2,11800,2009-12-10,162.0
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",6.9,4500,2007-05-19,169.0
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,6.3,4466,2015-10-26,148.0
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,7.6,9106,2012-07-16,165.0
4,49529,John Carter,"John Carter is a war-weary, former military ca...",6.1,2124,2012-03-07,132.0


In [4]:
print("Shape:", movies.shape)
print("\nColumns:")
print(movies.columns)
print("\nMissing values:")
print(movies.isna().sum())

Shape: (4803, 7)

Columns:
Index(['id', 'title', 'tags', 'vote_average', 'vote_count', 'release_date',
       'runtime'],
      dtype='str')

Missing values:
id              0
title           0
tags            0
vote_average    0
vote_count      0
release_date    1
runtime         2
dtype: int64


In [5]:
movies["tags"] = movies["tags"].fillna("")
movies["tags"] = movies["tags"].str.lower()

In [6]:
movies[["title", "tags"]].head()

,title,tags
0,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,Spectre,a cryptic message from bond’s past sends him o...
3,The Dark Knight Rises,following the death of district attorney harve...
4,John Carter,"john carter is a war-weary, former military ca..."


In [7]:
movies.loc[0, "tags"][:500]

'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron'

In [8]:
from sklearn.feature_extraction.text import CountVectorizer

In [9]:
count_vectorizer = CountVectorizer(max_features=5000, stop_words="english")

In [10]:
count_matrix = count_vectorizer.fit_transform(movies["tags"])

In [11]:
count_vectorizer.get_feature_names_out()[:20]

array(['000', '007', '10', '100', '11', '12', '13', '14', '15', '16',
       '17', '18', '18th', '19', '1930s', '1940s', '1944', '1950',
       '1950s', '1960s'], dtype=object)

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [13]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")

In [14]:
tfidf_matrix = tfidf_vectorizer.fit_transform(movies["tags"])

In [15]:
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (4803, 5000)


In [16]:
tfidf_vectorizer.get_feature_names_out()[:20]

array(['000', '007', '10', '100', '11', '12', '13', '14', '15', '16',
       '17', '18', '18th', '19', '1930s', '1940s', '1944', '1950',
       '1950s', '1960s'], dtype=object)

In [17]:
from sklearn.metrics.pairwise import cosine_similarity

In [18]:
similarity_matrix = cosine_similarity(tfidf_matrix)

In [19]:
print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (4803, 4803)


In [20]:
similarity_matrix[0][:10]

array([1.        , 0.021924  , 0.01218857, 0.0210417 , 0.10064622,
       0.05000363, 0.0032249 , 0.08162023, 0.01965563, 0.02003283])

In [21]:
similarity_matrix[0][0]

np.float64(1.0)

In [22]:
movies[movies["title"].str.contains("Avatar", case=False, na=False)][["id", "title"]]

,id,title
0,19995,Avatar


In [23]:
movie_index = movies[movies["title"] == "Avatar"].index[0]
movie_index

0

In [24]:
similarity_scores = list(enumerate(similarity_matrix[movie_index]))
similarity_scores[:5]

[(0, np.float64(1.0)),
 (1, np.float64(0.021924001088985574)),
 (2, np.float64(0.012188568103063789)),
 (3, np.float64(0.02104169907203823)),
 (4, np.float64(0.10064622200406348))]

In [25]:
sorted_scores = sorted(similarity_scores, key=lambda item: item[1], reverse=True)
sorted_scores[:10]

[(0, np.float64(1.0)),
 (3724, np.float64(0.20354233646724276)),
 (582, np.float64(0.19410894501161607)),
 (3604, np.float64(0.1839084762989157)),
 (47, np.float64(0.17110990997739733)),
 (539, np.float64(0.1653712943705198)),
 (942, np.float64(0.1626161410424448)),
 (2403, np.float64(0.15907608846978896)),
 (1914, np.float64(0.15809235389092297)),
 (557, np.float64(0.15563408328380338))]

In [26]:
top_similar_indices = [item[0] for item in sorted_scores[1:11]]
top_similar_indices

[3724, 582, 3604, 47, 539, 942, 2403, 1914, 557, 260]

In [27]:
movies.iloc[top_similar_indices][["title", "vote_average", "vote_count"]]

,title,vote_average,vote_count
3724,Falcon Rising,5.5,71
582,Battle: Los Angeles,5.5,1448
3604,Apollo 18,5.0,356
47,Star Trek Into Darkness,7.4,4418
539,Titan A.E.,6.3,313
942,The Book of Life,7.3,755
2403,Aliens,7.7,3220
1914,Lifeforce,6.2,135
557,Jarhead,6.6,765
260,Ender's Game,6.6,2303


In [28]:
def recommend_movies(title, movies_df, similarity_matrix, top_n=10):
    matching_movies = movies_df[movies_df["title"] == title]

    if matching_movies.empty:
        return pd.DataFrame()

    movie_index = matching_movies.index[0]
    similarity_scores = list(enumerate(similarity_matrix[movie_index]))

    sorted_scores = sorted(
        similarity_scores,
        key=lambda item: item[1],
        reverse=True
    )

    top_movie_indices = [item[0] for item in sorted_scores[1: top_n + 1]]

    recommendations = movies_df.iloc[top_movie_indices][
        ["id", "title", "vote_average", "vote_count", "release_date", "runtime"]
    ].copy()

    recommendations["similarity_score"] = [
        item[1] for item in sorted_scores[1: top_n + 1]
    ]

    return recommendations

In [29]:
recommend_movies("Avatar", movies, similarity_matrix, top_n=10)

,id,title,vote_average,vote_count,release_date,runtime,similarity_score
3724,270938,Falcon Rising,5.5,71,2014-09-05,103.0,0.203542
582,44943,Battle: Los Angeles,5.5,1448,2011-03-08,116.0,0.194109
3604,50357,Apollo 18,5.0,356,2011-07-20,86.0,0.183908
47,54138,Star Trek Into Darkness,7.4,4418,2013-05-05,132.0,0.171110
539,7450,Titan A.E.,6.3,313,2000-06-16,94.0,0.165371
942,228326,The Book of Life,7.3,755,2014-10-01,95.0,0.162616
2403,679,Aliens,7.7,3220,1986-07-18,137.0,0.159076
1914,11954,Lifeforce,6.2,135,1985-06-20,116.0,0.158092
557,25,Jarhead,6.6,765,2005-11-04,125.0,0.155634
260,80274,Ender's Game,6.6,2303,2013-10-23,114.0,0.154630


In [30]:
def recommend_movies(title, movies_df, similarity_matrix, top_n=10):
    normalized_title = title.lower()

    matching_movies = movies_df[
        movies_df["title"].str.lower() == normalized_title
    ]

    if matching_movies.empty:
        return pd.DataFrame()

    movie_index = matching_movies.index[0]
    similarity_scores = list(enumerate(similarity_matrix[movie_index]))

    sorted_scores = sorted(
        similarity_scores,
        key=lambda item: item[1],
        reverse=True
    )

    top_movie_indices = [item[0] for item in sorted_scores[1: top_n + 1]]

    recommendations = movies_df.iloc[top_movie_indices][
        ["id", "title", "vote_average", "vote_count", "release_date", "runtime"]
    ].copy()

    recommendations["similarity_score"] = [
        item[1] for item in sorted_scores[1: top_n + 1]
    ]

    return recommendations

In [31]:
recommend_movies("avatar", movies, similarity_matrix, top_n=10)

,id,title,vote_average,vote_count,release_date,runtime,similarity_score
3724,270938,Falcon Rising,5.5,71,2014-09-05,103.0,0.203542
582,44943,Battle: Los Angeles,5.5,1448,2011-03-08,116.0,0.194109
3604,50357,Apollo 18,5.0,356,2011-07-20,86.0,0.183908
47,54138,Star Trek Into Darkness,7.4,4418,2013-05-05,132.0,0.171110
539,7450,Titan A.E.,6.3,313,2000-06-16,94.0,0.165371
942,228326,The Book of Life,7.3,755,2014-10-01,95.0,0.162616
2403,679,Aliens,7.7,3220,1986-07-18,137.0,0.159076
1914,11954,Lifeforce,6.2,135,1985-06-20,116.0,0.158092
557,25,Jarhead,6.6,765,2005-11-04,125.0,0.155634
260,80274,Ender's Game,6.6,2303,2013-10-23,114.0,0.154630


In [32]:
recommend_movies("AVATAR", movies, similarity_matrix, top_n=10)

,id,title,vote_average,vote_count,release_date,runtime,similarity_score
3724,270938,Falcon Rising,5.5,71,2014-09-05,103.0,0.203542
582,44943,Battle: Los Angeles,5.5,1448,2011-03-08,116.0,0.194109
3604,50357,Apollo 18,5.0,356,2011-07-20,86.0,0.183908
47,54138,Star Trek Into Darkness,7.4,4418,2013-05-05,132.0,0.171110
539,7450,Titan A.E.,6.3,313,2000-06-16,94.0,0.165371
942,228326,The Book of Life,7.3,755,2014-10-01,95.0,0.162616
2403,679,Aliens,7.7,3220,1986-07-18,137.0,0.159076
1914,11954,Lifeforce,6.2,135,1985-06-20,116.0,0.158092
557,25,Jarhead,6.6,765,2005-11-04,125.0,0.155634
260,80274,Ender's Game,6.6,2303,2013-10-23,114.0,0.154630


In [33]:
test_titles = [
    "Avatar",
    "The Dark Knight Rises",
    "Toy Story",
    "The Godfather",
    "Titanic",
]

In [34]:
for title in test_titles:
    print(f"\nRecommendations for: {title}")
    display(recommend_movies(title, movies, similarity_matrix, top_n=5))


Recommendations for: Avatar


,id,title,vote_average,vote_count,release_date,runtime,similarity_score
3724,270938,Falcon Rising,5.5,71,2014-09-05,103.0,0.203542
582,44943,Battle: Los Angeles,5.5,1448,2011-03-08,116.0,0.194109
3604,50357,Apollo 18,5.0,356,2011-07-20,86.0,0.183908
47,54138,Star Trek Into Darkness,7.4,4418,2013-05-05,132.0,0.171110
539,7450,Titan A.E.,6.3,313,2000-06-16,94.0,0.165371



Recommendations for: The Dark Knight Rises


,id,title,vote_average,vote_count,release_date,runtime,similarity_score
65,155,The Dark Knight,8.2,12002,2008-07-16,152.0,0.456843
428,364,Batman Returns,6.6,1673,1992-06-19,126.0,0.398295
119,272,Batman Begins,7.5,7359,2005-06-10,140.0,0.349227
299,414,Batman Forever,5.2,1498,1995-05-31,121.0,0.338198
1359,268,Batman,7.0,2096,1989-06-23,126.0,0.304227



Recommendations for: Toy Story


,id,title,vote_average,vote_count,release_date,runtime,similarity_score
42,10193,Toy Story 3,7.6,4597,2010-06-16,103.0,0.513515
343,863,Toy Story 2,7.3,3806,1999-10-30,92.0,0.488322
1779,6957,The 40 Year Old Virgin,6.2,1983,2005-08-11,116.0,0.344450
3379,12271,Factory Girl,6.2,81,2006-12-29,90.0,0.178840
3873,11564,Class of 1984,6.2,66,1982-08-20,98.0,0.174300



Recommendations for: The Godfather


,id,title,vote_average,vote_count,release_date,runtime,similarity_score
2731,240,The Godfather: Part II,8.3,3338,1974-12-20,200.0,0.246981
867,242,The Godfather: Part III,7.1,1546,1990-12-24,162.0,0.233833
2464,13908,The Master of Disguise,3.7,78,2002-08-02,80.0,0.174498
3727,29920,Easy Money,6.5,63,2010-01-15,124.0,0.155080
1225,10154,Mickey Blue Eyes,5.3,140,1999-08-16,102.0,0.151016



Recommendations for: Titanic


,id,title,vote_average,vote_count,release_date,runtime,similarity_score
2143,9645,Ghost Ship,5.3,531,2002-10-25,91.0,0.219925
104,503,Poseidon,5.5,583,2006-05-12,99.0,0.209625
310,205775,In the Heart of the Sea,6.5,1276,2015-11-20,122.0,0.196038
818,109424,Captain Phillips,7.6,2454,2013-10-10,134.0,0.190324
17,1865,Pirates of the Caribbean: On Stranger Tides,6.4,4948,2011-05-14,136.0,0.188116


## Initial Recommendation Quality Notes

### What Works

The first TF-IDF content-based recommender can return movies that share similar content signals such as genres, keywords, cast, director, and overview terms.

This is a useful baseline because it does not require user ratings or user history.

It can recommend movies even for new users, as long as the movie has metadata.

### Current Limitations

The recommender depends heavily on exact words in the `tags` column.

It may miss semantic similarity when two movies use different words to describe similar ideas.

For example:

- `car` and `automobile`
- `space travel` and `interplanetary journey`
- `fear` and `terror`

The recommender also uses exact title matching after lowercasing, so it does not yet handle typos, partial search, or alternate titles.

The similarity scores are based only on content metadata, not on actual user preferences.

### Evaluation Notes

This stage uses qualitative evaluation by manually checking recommendations for several movie types.

Qualitative evaluation is useful early because it helps us understand model behavior before building formal metrics.

Later, we can compare models using stronger evaluation strategies such as user ratings, train-test splits, precision at K, recall at K, and ranking metrics.

### Next Improvements

Possible improvements include:

- Better title search
- Popularity-aware reranking
- Saving the model artifacts
- Moving recommender logic into reusable Python modules
- Building a Streamlit interface
- Comparing TF-IDF with advanced embeddings later

In [35]:
from src.recommenders.content_based import normalize_text

ModuleNotFoundError: No module named 'src'

In [ ]:
test_series = pd.Series(["Avatar", "THE MATRIX", None])
normalize_text(test_series)